# 38b · Arm EHR của `37` trên backbone **đã fine-tune** — và lần đầu **đo được nhiễu**

Backbone đông cứng trở lại ⇒ **một forward dùng chung** ⇒ mỗi arm chỉ ~15% chi phí ⇒
**chạy được nhiều seed**. Đây là chỗ đầu tiên trong cả Stage 4 mà thang nhiễu mua được với giá phải
chăng — và không có nó thì mọi con số `+0.003` của `37` vẫn treo.

## Hai backbone, **cùng một vòng lặp**

| | nguồn | vai trò |
|---|---|---|
| `pre` | RAD-DINO pretrain | tái lập bối cảnh của `37` — đo **trong** notebook này |
| `ft` | `38a / a2_E3 / backbone_ft.pt` | EHR sau khi ảnh đã tối đa |

```
G_ehr tren `ft`                       "anh da toi da roi, EHR CON them gi"   <- PHEP THU KHO NHAT
G_ehr(ft) - G_ehr(pre)                "fine-tune lam EHR bot hay them huu ich"
```

Có `pre` để câu thứ hai trả lời được **trong cùng notebook**, không phải so với `37` qua Drive —
đúng bài học đã trả giá ba lần (`mean_lin`, `mldec` v2-vs-v3, mốc `+0.101`).

## Backbone lấy theo QUY TẮC, không theo đường dẫn cứng

**Lấy lần chạy `38a` có điểm ủ xong (epoch cuối) cao nhất**, không lấy đỉnh giữa chừng:

```
a1_full_lr10   ft epoch cuoi 0.2231   (dinh 0.2395@ep3, CHUA u)
a2_E3          ft epoch cuoi 0.2412   <- dang dan dau
a4_E6_lr5e4    dinh ep2 roi di xuong  -> epoch cuoi se thap hon
```

> **Món nợ phải ghi:** `E=3` của `a2` được chọn bằng cách nhìn đường cong **val** của `a1`, và `a2`
> chưa ghi cột lát chọn. `a4` — lần đầu có cột đó — cho thấy lát chọn và val **đồng ý** về epoch
> đỉnh, nên lệch không lớn. Nhưng nó vẫn phải nằm trong `REPORT`.

**Chỉ lấy `bb`.** `head` trong file không dùng: nếu `mldec` khởi tạo từ đó còn 11 arm EHR khởi tạo
ngẫu nhiên thì `mldec` được lợi thế xuất phát và mọi `arm − mldec` lệch ngay từ bước 0.

## 12 arm, **verbatim `37`**

`C3`/`C4` đọc thẳng từ `37_cxrlt_mldecoder_fusion.ipynb` — cùng `EHRTok` · `EHRColTok` · `Tier1` ·
`MLDecoder` · `ASL`, cùng ba trục `tok` × `mode` × `t1`, cùng hai cặp đối chứng khớp tham số
(`gate_zero`, `t1_zero`). Không sửa một dấu nào, để so được với `37`.

## Cái thật sự mới: **biên độ ghép cặp**

Các arm dùng chung một forward, chung thứ tự dữ liệu, chung phép tăng cường — chỉ tham số riêng.
Nên `arm − mldec` là **hiệu ghép cặp**, phương sai nhỏ hơn hẳn phương sai của từng con số tuyệt đối.
`C8` báo `trung bình ± biên độ` cho **cả hai**, và chỉ hiệu ghép cặp mới được đọc.

## Ba điều kiện dừng, đặt TRƯỚC

```
G_ehr(ft) duong VA vuot bien do ghep cap
    -> EHR song sot qua fine-tune. Phat bieu manh nhat du an co the dua ra.

G_ehr(ft) ~ 0, dau DOI giua cac seed
    -> nhanh EHR DONG o muc KIEN TRUC. Khong them token/tang/cong nao nua.
       Lever con lai duy nhat la BIEU DIEN (EmbeddingBag thay one-hot) — o `29b`, khong o day.

G_ehr(ft) < G_ehr(pre) ro ret
    -> backbone duoc hoc da tu tim ra thu EHR dang bu. Khep lai cau "anh gan nhu du theo dinh nghia".
```

Kỷ luật kế thừa: `asl40` · **epoch cuối**, cosine trải trên `E` · clip **từng arm** · cột **lát chọn**
để chọn `E` · cột nhị phân không z-score · **tráo nhân quả** cho mọi arm có EHR.

## C0 · Config

In [ ]:
import os, json, math, time, random, shutil, tarfile, io as _io, re, glob, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'b1_pre_vs_ft'
RES38B  = os.path.join(OUT_DIR, 'stage4_nb38b', RUN_TAG); os.makedirs(RES38B, exist_ok=True)
CKPT    = os.path.join(RES38B, 'ckpt');                   os.makedirs(CKPT, exist_ok=True)
LOCAL   = '/content/cxrlt_518'

# Quy tac: lay lan chay 38a co diem U XONG (epoch cuoi) cao nhat, KHONG lay dinh giua chung.
BB_RUN    = 'a2_E3'
BACKBONES = ('pre', 'ft')       # bo 'pre' neu thieu VRAM/thoi gian; khi do mat phep so noi bo
SEEDS     = (42, 43)            # arm dung chung forward => `arm - mldec` la hieu GHEP CAP:
                                # 2 seed du de biet DAU co giu, chua du de trich bien do tin cay.

W_FRONTAL, W_LATERAL = 0.7, 0.3
SEED    = 42
SIZE    = 518
EPOCHS  = 8
BATCH, ACCUM = 120, 2           # EFF = 240, GIU NGUYEN qua 37 / 38a / 38b
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
G_LR_MULT = 20                  # LR rieng cho cong g_c (37: g_c gan nhu khong roi khoi 0 neu khong)
NW = min(8, (os.cpu_count() or 2))
NOISE_MACRO, NOISE_TAIL = 0.0007, 0.0023

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0.'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'thiet bi: {torch.cuda.get_device_name(0)} | VRAM {_vram:.0f} GB | AMP {AMP}')

BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
_all38a = sorted(os.listdir(os.path.join(OUT_DIR, 'stage4_nb38a'))) \
          if os.path.isdir(os.path.join(OUT_DIR, 'stage4_nb38a')) else []
assert os.path.exists(BB_SRC), f'khong thay {BB_SRC}. 38a co san: {_all38a}'

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head', 'medium', 'tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V = ((pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
     else ORDER[['dicom_id', 'view_group']].assign(use=lambda d: d.view_group != 'unknown'))

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL           # CUNG cach cat nhu 35/37/38a
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA = np.where(TF_)[0], np.where(SL)[0], np.where(VA)[0]
N_POS_TF = YM[TF_].sum(0)

print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'ghi vao: {RES38B}')
print(f'backbone: {BACKBONES} | nguon `ft` = 38a/{BB_RUN} | seeds {SEEDS} | E={EPOCHS} '
      f'| BATCH {BATCH}x{ACCUM} = EFF {BATCH*ACCUM} | NW {NW}')

## C1 · Nạp EHR — **verbatim `37`**

Cùng hai tập `Q_CTX`/`Q_FULL`, cùng chín họ cột, cùng bản sửa **cột nhị phân không z-score**
(one-hot `micro_org_*` cực hiếm làm z-score cho ra `468`, mà attention có score tỉ lệ `M²`).

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()

def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = {'Q_CTX': [c for c in SPEC['Q_CTX'] if c in E_DF.columns],
        'Q_FULL': [c for c in SPEC['Q_FULL'] if c in E_DF.columns]}
EHR = {k: _mat(v) for k, v in COLS.items()}
# chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val
EHR_MU, EHR_SD = {}, {}
EHR_CLIP = 10.0     # cot NHI PHAN cuc hiem lam chuan hoa no ra gia tri KHONG LO:
                    #   1 duong / 219,785  ->  sd 2.1e-3  ->  gia tri chuan hoa 468
                    # Probe tuyen tinh (35) chiu duoc — chi can hoc trong so nho. ATTENTION thi
                    # KHONG: `EHRColTok` dua thang x_j vao token (T_j = b_j + x_j*W_j), va score
                    # ti le voi M^2 nen bien do 30 cho score ~2e4 -> softmax bao hoa -> inf/NaN.
BIN_OF = {}
for k, m in EHR.items():
    # Cot NHI PHAN thi KHONG z-score. z-score bien gia tri "co" thanh sqrt((1-p)/p):
    #   p = 1/219,785 -> 468   |   p = 0.01 -> 9.9   |   p = 0.5 -> 1.0
    # Do chinh la nguon cua gia tri khong lo, va no la LOI BIEU DIEN chu khong phai
    # loi cua du lieu: cot chi bao khong co "do lech chuan" de chia.
    # `29b` one-hot top-K vi khuan (`micro_org_*`) + co comorbidity => rat nhieu cot cuc hiem.
    b_ = np.array([np.isin(np.unique(m[:, j]), (0.0, 1.0)).all() for j in range(m.shape[1])])
    BIN_OF[k] = b_
    mu = m[TF_].mean(0, keepdims=True); sd = m[TF_].std(0, keepdims=True) + 1e-6
    mu[:, b_] = 0.0; sd[:, b_] = 1.0                       # nhi phan: giu nguyen 0/1
    EHR_MU[k], EHR_SD[k] = mu, sd
    z = (m - mu) / sd
    _n_out = int((np.abs(z) > EHR_CLIP).sum())
    EHR[k] = np.clip(z, -EHR_CLIP, EHR_CLIP).astype('float32')  # kep chi con la chan cuoi
    print(f'  {k}: {int(b_.sum())}/{len(b_)} cot NHI PHAN (giu 0/1) | '
          f'|z| max {np.abs(z).max():6.1f} | kep {_n_out:,} o ({_n_out/z.size:.4%})')

# --- gom cot thanh 8 HO. Quy tac, khong phai tay; moi cot phai roi vao DUNG mot ho. ---
FAM_RE = [('ctx',    r'^ctx_'),
          ('demo',   r'^(age|gender)$'),   # nhan khau hoc: LUON co mat, khong phai phep do.
                                           # `29b` xep vao "gia tri" vi muc dich cua no la hoi
                                           # "loai benh nhan vs benh"; o day chung la HO rieng.
          ('mask',   r'_mask$'),
          ('n_lab',  r'^n_lab_'),
          ('dh_lab', r'^dh_lab_'),
          ('lab',    r'^lab_'),
          ('vital',  r'^vital_'),
          ('como',   r'^como_'),
          ('micro',  r'^micro_')]
def famsplit(cols):
    out, seen = {}, set()
    for nm, rx in FAM_RE:
        g = [i for i, c in enumerate(cols) if re.search(rx, c) and i not in seen]
        seen |= set(g)
        if g: out[nm] = np.array(g)
    du = [c for i, c in enumerate(cols) if i not in seen]
    assert not du, f'cot khong thuoc ho nao: {du[:10]}'
    return out
FAM = {k: famsplit(v) for k, v in COLS.items()}
for k in COLS:
    print(f'{k}: {len(COLS[k])} cot -> {len(FAM[k])} ho  ' +
          ' | '.join(f'{n}:{len(ix)}' for n, ix in FAM[k].items()))

## C2 · Ảnh local + **hai** backbone đông cứng

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_have:,} / {len(ORDER):,}')
if _have < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        name, blob = pl
        dst = os.path.join(LOCAL, os.path.basename(name))
        if os.path.exists(dst): return 0
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95); return 1
    t0 = time.time()
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _have = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
    print(f'stage xong: {_have:,} anh | {(time.time()-t0)/60:.1f} phut')
assert _have >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_have:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

BB = {}
if 'pre' in BACKBONES:
    BB['pre'] = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
if 'ft' in BACKBONES:
    _c = torch.load(BB_SRC, map_location='cpu', weights_only=False)
    m = AutoModel.from_pretrained('microsoft/rad-dino')
    _miss = m.load_state_dict(_c['bb'], strict=True)
    BB['ft'] = m.to(dev).eval()
    print(f'BACKBONE `ft`: 38a/{BB_RUN}  E={_c.get("E")}, epoch cuoi={_c.get("ep")}, '
          f'ft-lp={_c.get("ft_lp", float("nan")):+.4f}')
    print('  CHI lay `bb`. `head` trong file KHONG dung — 12 arm deu khoi tao ngau nhien cung seed,')
    print('  neu khong `mldec` co loi the xuat phat va moi `arm - mldec` lech tu buoc 0.')
    print(f'  LUU Y: `E` cua lan chay do chon bang duong cong VAL cua a1 (a2 chua ghi cot lat chon).')
for k, m in BB.items():
    for p in m.parameters(): p.requires_grad_(False)
    assert not any(p.requires_grad for p in m.parameters()), f'{k} chua dong cung'
D_CXR = BB[BACKBONES[0]].config.hidden_size
backbone = BB[BACKBONES[0]]      # bi danh cho cong `assert` cua C4 (doc verbatim tu 37)

# --- hai backbone PHAI khac nhau, neu khong `ft` khong duoc nap ---
if len(BB) == 2:
    with torch.no_grad():
        _d = max((a - b).abs().max().item()
                 for a, b in zip(BB['pre'].state_dict().values(), BB['ft'].state_dict().values()))
    assert _d > 1e-6, f'bb_pre va bb_ft GIONG HET nhau (lech {_d:.1e}) -> `ft` khong duoc nap'
    print(f'cong: hai backbone KHAC nhau (lech toi da {_d:.3f}) — `ft` da duoc nap dung')

class DS(Dataset):
    '''`ehr_perm`: hoan vi hang EHR cho phep TRAO NHAN QUA. Nhan y KHONG di theo hoan vi.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        j = i if s.perm is None else int(s.perm[k])
        return dict(image=s.tf(Image.merge('RGB', (im, im, im))),
                    y=torch.from_numpy(YM[i]), row=i, erow=j)

def loader(idx, tf, shuffle, bs, perm=None, nw=NW):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

@torch.no_grad()
def fwd_all(img):
    '''MOT forward moi backbone, dung chung cho ca 12 arm cua backbone do.'''
    out = {}
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        for k, m in BB.items(): out[k] = m(pixel_values=img).last_hidden_state[:, 1:].float()
    return out

_b = next(iter(loader(I_VA[:8], eval_tf, False, 8, nw=0)))
_p = fwd_all(_b['image'].to(dev))
N_TOK = _p[BACKBONES[0]].shape[1]
print(f'patch {tuple(_p[BACKBONES[0]].shape)} <- {N_TOK} token | {len(BB)} backbone')
assert N_TOK == (SIZE // BB[BACKBONES[0]].config.patch_size) ** 2

## C3 · Kiến trúc — **verbatim `37`**

`GroupFC` · `XAttn` · `MLDecoder` · `EHRTok` · `EHRColTok` · `Tier1` · `MeanLin` · `ASL`.
Không sửa một dấu nào, để mọi so sánh với `37` đọc được.

In [ ]:
class GroupFC(nn.Module):
    '''Moi lop mot vector trong so rieng -> 1 logit. Khong chia se giua cac lop.'''
    def __init__(s, C, d):
        super().__init__(); s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b        # [B, C, d] -> [B, C]

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a)
        return s.n2(h + s.ff(h)), w

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = chuoi token.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, ehr='none', d_ehr_in=0, K=8,
                 t1=False):
        super().__init__()
        s.mode = ehr; s.K = K
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d)
        s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None      # B2a — dat TRUOC tang 2
        if ehr != 'none' or t1:
            s.ein = nn.Linear(d_ehr_in, d)
            if ehr == 'gate':
                s.ehr = XAttn(d, nhead, ff, drop)
                s.pe  = nn.Linear(d, d)                # KHOI TAO BINH THUONG — xem ghi chu duoi
                s.g = nn.Parameter(torch.zeros(C))     # cong TUNG LOP, init 0 (mau ReZero)
                # KHONG zero-init CA HAI. `h + g*pe(he)` voi g=0 VA pe=0 la diem yen ngua chet:
                #   dL/dg  = pe(he) = 0   va   dL/dpe = g = 0   -> nhanh EHR khong bao gio chay.
                # Dry-run da bat dung loi nay (g_c khong nhuc nhich sau 3 epoch).
                # Chi `g=0` la du de khoi dong DUNG BANG CXR-only, va khi do dL/dg = pe(he) != 0.
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        B = patch.shape[0]
        q  = s.q.weight.unsqueeze(0).expand(B, -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None:                              # TANG 1: patch duoc EHR dieu bien
            kv = s.t1(kv, s.ein(etok))                    # theo VUNG, truoc khi query lop doc
        if s.mode == 'cat':
            kv = torch.cat([kv, s.ein(etok)], 1)                 # softmax tren 1369+K
            h, w = s.img(q, kv, need_w); return s.out(h), w
        h, w = s.img(q, kv, need_w)
        if s.mode == 'gate':
            he, we = s.ehr(q, s.ein(etok), need_w)               # softmax CHI tren K token EHR
            h = h + s.g.view(1, -1, 1) * s.pe(he)
            return s.out(h), (w, we)
        return s.out(h), w

class EHRTok(nn.Module):
    '''K token EHR. `fam`: dict ho -> chi so cot. mode `zero` = vector HANG SO hoc duoc,
       khop tham so voi ban that (doi chung H4z/H6z cua 06b).'''
    def __init__(s, fam, d=256, const=False):
        super().__init__()
        s.fam, s.const = fam, const
        s.enc = nn.ModuleList([nn.Sequential(nn.Linear(len(ix), d), nn.GELU(), nn.Linear(d, d))
                               for ix in fam.values()])
        s.idx = [torch.as_tensor(ix) for ix in fam.values()]
        if const: s.c = nn.Parameter(torch.randn(len(fam), d) * 0.02)
        s.d = d
    def forward(s, e):
        if s.const: return s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        return torch.stack([m(e[:, ix.to(e.device)]) for m, ix in zip(s.enc, s.idx)], 1)

class EHRColTok(nn.Module):
    '''247 token COT (FT-Transformer: T_j = b_j + x_j*W_j) + 1 token TONG HOP tu MLP nhin
       CA 247 cot cung luc.

       Tinh chat quyet dinh: chuoi nay CHUA k1 nhu TRUONG HOP RIENG — attention chi can don
       het khoi luong len token tong hop la tai lap duoc k1. `EHRTok` theo HO (K=9) KHONG co
       tinh chat do (khong ho nao chua ca 247 cot), va do chinh la ly do no THUA k1 o v1.

       Gop dung hai thu ma K=1 va K=9 moi ben chi co mot:
         token cot     -> CHON LOC tung cot ("Pneumonia doc cot bach cau", khong phai ca ho)
         token tong hop-> TUONG TAC CHEO (attention la to hop LOI, khong tao duoc tich cot)

       HUONG A: moi cot LUON co token. Cot thieu -> (0-mu)/sigma, mot DIEM CO DINH tren duong
       thang cua cot do, mo hinh cham diem thap cho no duoc. Khong `key_padding_mask` => khong
       co kieu loi 'hang toan -inf -> NaN' voi 11,839 study khong co ban ghi nao.'''
    def __init__(s, d_in, d=256, const=False):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d)   # chan bien do token: `col` dua thang x_j vao token nen mot cot
                                 # hiem co the lan at 247 token con lai. `EHRTok` khong can vi
                                 # gia tri cua no da di qua GELU.
        s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)   # doi chung: HANG SO hoc duoc
    def forward(s, e):
        if s.const: return s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)     # [B, 247, d]
        return s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))       # [B, 248, d]

class Tier1(nn.Module):
    '''B2a — Q = 1369 PATCH, K/V = token EHR. FiLM BIEN THIEN THEO KHONG GIAN.

       softmax chi tren token EHR nen cac patch KHONG canh tranh voi nhau: moi patch tu hoi
       EHR rieng, va EHR khong the bi 1369 patch nhan chim (do la co che that bai cua 06b H6).
       Tra loi 'boi canh khien VUNG NAO dang ngo' — truc KHONG GIAN, khac han truc NGU NGHIA
       cua tang 2 ('NHAN NAY co quan tam EHR khong').

       `pr` ZERO-INIT => khoi dong DUNG BANG CXR-only. CHI MOT he so zero-init: khac v1, noi
       `g_c=0` VA `pe=0` tao diem yen ngua chet (dL/dg = pe(he) = 0 va dL/dpe ~ g = 0).
       O day dL/dpr = dL/dpatch' != 0 ngay tu buoc dau.

       Chi phi: 1369 x 248 attention = 18% mot lop self-attention (1369^2 = 1.87M).
       `need_weights=False` cho phep duong SDPA hop nhat -> khong hien thuc hoa ma tran [1369, 248].'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n  = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(a)

class MeanLin(nn.Module):
    def __init__(s, C, d_in, d_ehr=0):
        super().__init__(); s.fc = nn.Linear(d_in + d_ehr, C); s.use_e = d_ehr > 0
    def forward(s, patch, e=None):
        h = patch.mean(1)
        return s.fc(torch.cat([h, e], 1) if s.use_e else h), None

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

## C4 · 12 arm × **mỗi backbone** + cổng `assert`

`ARMS` và `build_arms` lấy verbatim `37`; ở đây gọi `build_arms` **một lần cho mỗi backbone**, cùng
`seed`, nên hai bản khởi tạo **giống hệt nhau** — `pre` vs `ft` chỉ khác đặc trưng đầu vào.

In [ ]:
D_MLD, D_ETOK = 512, 256
# Ba TRUC doc lap, dung lan nhau:
#   tok  : chuoi K/V EHR gom gi     fam(9 ho) | k1(1 vector) | col(247 cot + 1 tong hop)   <- B2b
#   mode : query LOP doc EHR the nao  none | cat([patch;ehr]) | gate(attention rieng + g_c)  <- B2c
#   t1   : PATCH co duoc EHR dieu bien theo vung khong                                        <- B2a
ARMS = {
  'mean_lin':        dict(head='mean', ehr=None),
  'mean_lin_ehr':    dict(head='mean', ehr='Q_CTX'),
  'mldec':           dict(head='mld',  ehr=None,     mode='none'),
  # --- v1: giu nguyen de doi chieu ---
  'mldec_cat':       dict(head='mld',  ehr='Q_FULL', mode='cat',  tok='fam'),
  'mldec_gate':      dict(head='mld',  ehr='Q_FULL', mode='gate', tok='fam'),
  'mldec_gate_zero': dict(head='mld',  ehr='Q_FULL', mode='gate', tok='fam', const=True),
  'mldec_gate_k1':   dict(head='mld',  ehr='Q_FULL', mode='gate', tok='k1'),
  # --- huong A: 247 cot + 1 tong hop (B2b) ---
  'mldec_col_cat':   dict(head='mld',  ehr='Q_FULL', mode='cat',  tok='col'),
  'mldec_col_gate':  dict(head='mld',  ehr='Q_FULL', mode='gate', tok='col'),
  # --- B2a: tang 1, Q = patch ---
  'mldec_t1':        dict(head='mld',  ehr='Q_FULL', mode='none', tok='col', t1=True),
  'mldec_t1_zero':   dict(head='mld',  ehr='Q_FULL', mode='none', tok='col', t1=True, const=True),
  'mldec_t1t2':      dict(head='mld',  ehr='Q_FULL', mode='gate', tok='col', t1=True),
}
# `k1`: gop CA vector EHR thanh MOT token. Softmax tren 1 phan tu = 1.0 => moi query lop nhan
# CUNG mot vector => suy bien thanh BIAS TUNG LOP. Neu K=8 khong hon K=1 thi viec tach theo HO
# khong mua duoc gi, va bang [40 x 8] o C9-bis chi la hoa van.

def build_arms(seed=SEED):
    torch.manual_seed(seed)
    M, E = nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        if cfg['head'] == 'mean':
            M[tag] = MeanLin(C, D_CXR, EHR[cfg['ehr']].shape[1] if cfg['ehr'] else 0)
        else:
            K = 0
            if cfg['ehr']:
                n_in, tok, cst = EHR[cfg['ehr']].shape[1], cfg.get('tok', 'fam'), cfg.get('const', False)
                if tok == 'col':
                    E[tag] = EHRColTok(n_in, D_ETOK, cst);            K = n_in + 1
                else:
                    fam = {'all': np.arange(n_in)} if tok == 'k1' else FAM[cfg['ehr']]
                    E[tag] = EHRTok(fam, D_ETOK, cst);                K = len(fam)
            M[tag] = MLDecoder(C, D_CXR, D_MLD, ehr=cfg['mode'], d_ehr_in=D_ETOK, K=K,
                               t1=cfg.get('t1', False))
    return M.to(dev), E.to(dev)

MOD, ETK = build_arms()
print(f'{"arm":<17} {"tham so":>12}')
for tag in ARMS:
    n = sum(p.numel() for p in MOD[tag].parameters() if p.requires_grad)
    n += sum(p.numel() for p in ETK[tag].parameters()) if tag in ETK else 0
    print(f'  {tag:<15} {n:>12,}')

# --- cong 1: backbone khong co gradient nao ---
assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
# --- cong 2: query lop KHONG hoc (thiet ke cua bai goc, dung "sua") ---
for t in ('mldec', 'mldec_gate'):
    assert not MOD[t].q.weight.requires_grad, f'{t}: query_embed dang hoc'
# --- cong 3: gate zero-init => mldec_gate PHAI cho logit y het mldec_gate voi g=0 ---
with torch.no_grad():
    _p = torch.randn(4, N_TOK, D_CXR, device=dev)
    _e = torch.randn(4, EHR['Q_FULL'].shape[1], device=dev)
    _t = ETK['mldec_gate'](_e)
    MOD['mldec_gate'].eval()
    l1, _ = MOD['mldec_gate'](_p, _t)
    _g = MOD['mldec_gate'].g.clone(); MOD['mldec_gate'].g.data.fill_(0.0)
    l2, _ = MOD['mldec_gate'](_p, _t)
    MOD['mldec_gate'].g.data.copy_(_g)
    d = (l1 - l2).abs().max().item()
assert d < 1e-5, f'gate KHONG zero-init: lech {d:.2e} -> gain<0 se khong phan biet duoc EHR vo dung voi adapter gay nhieu'
print(f'\ncong: backbone dong cung OK | query co dinh OK | gate zero-init OK (lech {d:.1e})')

# --- cong 5: Tier1 zero-init => arm co t1 PHAI cho logit y het khi bo tang 1 ---
with torch.no_grad():
    _t = ETK['mldec_t1'](_e); MOD['mldec_t1'].eval()
    l3, _ = MOD['mldec_t1'](_p, _t)
    _keep = MOD['mldec_t1'].t1; MOD['mldec_t1'].t1 = None
    l4, _ = MOD['mldec_t1'](_p, _t)
    MOD['mldec_t1'].t1 = _keep
    d1 = (l3 - l4).abs().max().item()
assert d1 < 1e-5, f'Tier1 KHONG zero-init: lech {d1:.2e}'
print(f'cong: Tier1 zero-init OK (lech {d1:.1e}) — khoi dong dung bang CXR-only')

# --- cong 6: chuoi EHR cua arm `col` phai dai 247+1 ---
with torch.no_grad():
    _nc = ETK['mldec_col_gate'](_e).shape[1]
assert _nc == EHR['Q_FULL'].shape[1] + 1, f'chuoi col = {_nc}, phai la {EHR["Q_FULL"].shape[1]}+1'
print(f'cong: chuoi col = {_nc} token ({_nc-1} cot + 1 tong hop) OK — CHUA k1 nhu truong hop rieng')
# --- cong 7: doi chung khop tham so ---
n1 = sum(p.numel() for p in ETK['mldec_gate'].parameters())
n0 = sum(p.numel() for p in ETK['mldec_gate_zero'].parameters())
print(f'doi chung: gate {n1:,} vs gate_zero {n0:,} tham so EHR '
      f'({"KHOP" if abs(n1-n0)/max(n1,1) < 0.5 else "LECH — G se lan ca hieu ung dung luong"})')

# ================= 38b: mot ban arm cho MOI backbone, khoi tao GIONG HET =================
def build_all(seed=SEED):
    M, E = {}, {}
    for k in BACKBONES:
        M[k], E[k] = build_arms(seed)          # cung seed => hai ban giong het nhau
    return M, E

MODS, ETKS = build_all(SEED)
if len(BACKBONES) == 2:
    with torch.no_grad():
        _d = max((MODS['pre'].state_dict()[k] - MODS['ft'].state_dict()[k]).abs().max().item()
                 for k in MODS['pre'].state_dict())
    assert _d < 1e-6, f'arm cua hai backbone khoi tao KHAC nhau (lech {_d:.2e})'
    print(f'cong: arm cua `pre` va `ft` khoi tao GIONG HET (lech {_d:.1e})')
_n = sum(p.numel() for p in MODS[BACKBONES[0]].parameters() if p.requires_grad)
print(f'{len(ARMS)} arm x {len(BACKBONES)} backbone = {len(ARMS)*len(BACKBONES)} dau doc '
      f'| {_n:,} tham so moi ban')

## C5 · Chấm điểm — verbatim `32` C2 / `35` C3 / `37` C6

In [ ]:
from sklearn.metrics import average_precision_score
_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V.view_group.to_numpy()[VA] == 'frontal') & V.use.to_numpy()[VA]
_isL = (V.view_group.to_numpy()[VA] == 'lateral') & V.use.to_numpy()[VA]

def pool_study(p):
    out = np.zeros((len(_uid), p.shape[1]), 'float64'); ws = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(p.shape[1])], 1)
        has = cnt > 0
        out[has] += w * acc[has] / cnt[has, None]; ws[has] += w
    ok = ws > 0; out[ok] /= ws[ok, None]; return out, ok
_first = np.zeros(len(_uid), int); _first[_inv[::-1]] = np.arange(VA.sum())[::-1]
_Ys = YM[VA][_first]

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def score(p):
    ap_i = ap_cols(YM[VA], p)
    ps, ok = pool_study(p)
    ap_s = np.array([average_precision_score(_Ys[ok][:, j], ps[ok][:, j]) for j in range(C)])
    return ap_i, ap_s
print(f'val {VA.sum():,} anh -> {len(_uid):,} study')

## C6 · Vòng lặp — mỗi seed một lần chạy đầy đủ

Không early-stop, cosine trải trên `E`, lấy **epoch cuối**. Mỗi epoch chấm **lát chọn** (nơi duy nhất
được chọn `E`) và **val** (chỉ báo cáo). Clip **từng arm** — chuẩn toàn cục ≈ `√n ×` chuẩn một arm nên
clip chung sẽ ghép 24 đầu đọc lại với nhau, phá vỡ tương đương `AMENDMENTS` ①.

`RESUME` theo `(seed, epoch)`: đứt phiên chạy lại là tiếp.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)

def ehr_batch(M, E, tag, erow):
    cfg = ARMS[tag]
    if not cfg['ehr']: return None
    e = torch.from_numpy(EHR[cfg['ehr']][erow.numpy()]).to(dev)
    return e if cfg['head'] == 'mean' else E[tag](e)

def run_arms(M, E, patch, erow, need_w=False):
    out = {}
    for tag, cfg in ARMS.items():
        e = ehr_batch(M, E, tag, erow)
        out[tag] = M[tag](patch, e, need_w=need_w) if cfg['head'] == 'mld' else M[tag](patch, e)
    return out

@torch.no_grad()
def predict(MODS, ETKS, idx, perm=None, bs=64):
    for k in BACKBONES: MODS[k].eval(); ETKS[k].eval()
    P = {(k, t): np.zeros((len(idx), C), 'float32') for k in BACKBONES for t in ARMS}
    i = 0
    for b in tqdm(loader(idx, eval_tf, False, bs, perm, nw=0), desc='eval', leave=False):
        pt = fwd_all(b['image'].to(dev, non_blocking=True)); n = b['image'].shape[0]
        for k in BACKBONES:
            for t, (lg, _) in run_arms(MODS[k], ETKS[k], pt[k], b['erow']).items():
                P[(k, t)][i:i+n] = torch.sigmoid(lg.float()).cpu().numpy()
        i += n
    _bad = {kt: int(np.isnan(v).sum()) for kt, v in P.items() if np.isnan(v).any()}
    assert not _bad, f'predict cho ra NaN: {_bad}'
    return P

def make_opt(MODS, ETKS):
    '''LR rieng cho cong `g_c` (x20): 37 cho thay g_c gan nhu khong roi khoi 0 neu dung LR chung.'''
    gates, rest = [], []
    for k in BACKBONES:
        for t in ARMS:
            if getattr(MODS[k][t], 'mode', None) == 'gate': gates.append(MODS[k][t].g)
    gid = {id(p) for p in gates}
    for k in BACKBONES:
        rest += [p for m in (MODS[k], ETKS[k]) for p in m.parameters()
                 if p.requires_grad and id(p) not in gid]
    dec = [p for p in rest if p.ndim > 1]; nod = [p for p in rest if p.ndim <= 1]
    assert gates, 'khong tim thay g_c nao'
    return torch.optim.AdamW([{'params': dec,   'weight_decay': WD},
                              {'params': nod,   'weight_decay': 0.0},
                              {'params': gates, 'lr': LR * G_LR_MULT, 'weight_decay': 0.0}], lr=LR)

P_LOG = os.path.join(RES38B, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL  = loader(I_TF, train_tf, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'):
        print(f'{sk}: da xong, bo qua'); continue
    MODS, ETKS = build_all(seed)
    PARAM_OF = {(k, t): [p for m in ([MODS[k][t]] + ([ETKS[k][t]] if t in ETKS[k] else []))
                         for p in m.parameters() if p.requires_grad]
                for k in BACKBONES for t in ARMS}
    N_CLIP = {kt: 0 for kt in PARAM_OF}; N_STEP = 0
    opt = make_opt(MODS, ETKS)
    steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS; warm = int(WARMUP * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    for ep in range(EPOCHS):
        for k in BACKBONES: MODS[k].train(); ETKS[k].train()
        t0, tot, nb = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            pt = fwd_all(b['image'].to(dev, non_blocking=True)); y = b['y'].to(dev)
            loss = 0.0
            for k in BACKBONES:
                for t, (lg, _) in run_arms(MODS[k], ETKS[k], pt[k], b['erow']).items():
                    if not torch.isfinite(lg).all():
                        raise RuntimeError(f'{sk} ep{ep+1} buoc {j}: `{k}/{t}` logit NaN/inf')
                    loss = loss + crit(lg.float(), y)
            (loss / ACCUM).backward(); tot += float(loss.detach()); nb += 1
            if (j + 1) % ACCUM == 0:
                for kt, ps in PARAM_OF.items():
                    if float(torch.nn.utils.clip_grad_norm_(ps, CLIP)) > CLIP: N_CLIP[kt] += 1
                N_STEP += 1
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        PV = predict(MODS, ETKS, I_VA); PS = predict(MODS, ETKS, I_SL)
        for kt, p in PV.items(): np.save(os.path.join(RES38B, f'p_{kt[0]}_{kt[1]}_{sk}_ep{ep+1}.npy'), p)
        _e = {}
        for kt, p in PV.items():
            a = score(p)[0]
            _e[f'{kt[0]}/{kt[1]}'] = dict(mAP_anh=float(np.nanmean(a)), **tiers(a),
                                          sel=float(np.nanmean(ap_cols(YM[SL], PS[kt]))))
        LOG[sk][str(ep)] = _e; json.dump(LOG, open(P_LOG, 'w'))
        _b0 = BACKBONES[-1]
        print(f'{sk} ep{ep+1:>2} loss {tot/max(nb,1):.4f} | '
              + ' '.join(f'{k}/mldec {_e[f"{k}/mldec"]["mAP_anh"]:.4f}' for k in BACKBONES)
              + f' | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | {(time.time()-t0)/60:.1f} ph')
        print(f'   LAT CHON {_b0}: ' + ' '.join(
            f'{t.replace("mldec_","")}:{_e[f"{_b0}/{t}"]["sel"]:.4f}'
            for t in ('mldec', 'mldec_gate', 'mldec_gate_zero', 'mldec_col_cat'))
            + '   <- CHON `E` BANG COT NAY')
        _f = {kt: N_CLIP[kt]/max(N_STEP,1) for kt in PARAM_OF}
        if max(_f.values()) > 0.01:
            print('   clip kich hoat: ' + ' '.join(f'{k}/{t}:{v:.0%}' for (k,t),v in _f.items() if v>0.01))
    torch.save(dict(mods={k: MODS[k].state_dict() for k in BACKBONES},
                    etks={k: ETKS[k].state_dict() for k in BACKBONES}, seed=seed),
               os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong')
print('tat ca seed xong')

## C7 · Bảng chính + **biên độ ghép cặp**

Chỉ `arm − mldec` mới được đọc: các arm dùng chung một forward, chung thứ tự dữ liệu, chung phép
tăng cường, nên hiệu của chúng có phương sai nhỏ hơn hẳn từng con số tuyệt đối.

In [ ]:
LOG = json.load(open(P_LOG))
EPk = str(EPOCHS - 1)
rows = []
for seed in SEEDS:
    sk = f's{seed}'
    if EPk not in LOG.get(sk, {}): print(f'{sk}: chua co ep{EPOCHS}'); continue
    for kt, v in LOG[sk][EPk].items():
        k, t = kt.split('/', 1)
        rows.append(dict(seed=seed, bb=k, arm=t, **v))
R = pd.DataFrame(rows)
assert len(R), 'chua co ket qua nao'

print('=== TUYET DOI (TB qua seed) ===')
for k in BACKBONES:
    piv = (R[R.bb == k].pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel'],
                                    aggfunc='mean').reindex(list(ARMS)))
    piv['bien_do'] = R[R.bb == k].groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min()).reindex(list(ARMS))
    print(f'\n--- backbone `{k}` ---')
    print(piv.to_string(float_format=lambda v: f'{v:.4f}'))

print('\n=== HIEU GHEP CAP  arm - mldec  (day moi la cho duoc doc) ===')
D = []
for k in BACKBONES:
    for t in ARMS:
        if t == 'mldec': continue
        d = [R[(R.bb==k)&(R.arm==t)&(R.seed==s)].mAP_anh.iloc[0]
             - R[(R.bb==k)&(R.arm=='mldec')&(R.seed==s)].mAP_anh.iloc[0]
             for s in SEEDS if len(R[(R.bb==k)&(R.arm==t)&(R.seed==s)])]
        dt = [R[(R.bb==k)&(R.arm==t)&(R.seed==s)]['tail'].iloc[0]
              - R[(R.bb==k)&(R.arm=='mldec')&(R.seed==s)]['tail'].iloc[0]
              for s in SEEDS if len(R[(R.bb==k)&(R.arm==t)&(R.seed==s)])]
        if not d: continue
        D.append(dict(bb=k, arm=t, d_anh=np.mean(d), bien_do=max(d)-min(d),
                      cung_dau=all(x > 0 for x in d) or all(x < 0 for x in d),
                      d_tail=np.mean(dt), bien_do_tail=max(dt)-min(dt)))
D = pd.DataFrame(D)
print(D.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
print('\n  DOC: chi tin mot hieu khi `cung_dau` = True VA |d| > `bien_do`.')

print('\n=== G_ehr = gate - gate_zero  (doi chung KHOP THAM SO) ===')
for k in BACKBONES:
    g = [R[(R.bb==k)&(R.arm=='mldec_gate')&(R.seed==s)].mAP_anh.iloc[0]
         - R[(R.bb==k)&(R.arm=='mldec_gate_zero')&(R.seed==s)].mAP_anh.iloc[0]
         for s in SEEDS if len(R[(R.bb==k)&(R.arm=='mldec_gate')&(R.seed==s)])]
    gt = [R[(R.bb==k)&(R.arm=='mldec_gate')&(R.seed==s)]['tail'].iloc[0]
          - R[(R.bb==k)&(R.arm=='mldec_gate_zero')&(R.seed==s)]['tail'].iloc[0]
          for s in SEEDS if len(R[(R.bb==k)&(R.arm=='mldec_gate')&(R.seed==s)])]
    if g: print(f'  {k:<4} G_ehr {np.mean(g):+.4f} (bien do {max(g)-min(g):.4f}) | '
                f'tail {np.mean(gt):+.4f} (bien do {max(gt)-min(gt):.4f})')
if len(BACKBONES) == 2 and len(R):
    _gp = D[(D.bb=='pre')&(D.arm=='mldec_gate')].d_anh
    _gf = D[(D.bb=='ft')&(D.arm=='mldec_gate')].d_anh
    if len(_gp) and len(_gf):
        print(f'\n  G(ft) - G(pre) = {float(_gf.iloc[0]-_gp.iloc[0]):+.4f}')
        print('  AM  => backbone duoc hoc da tu tim ra thu EHR dang bu.')
        print('  DUONG => fine-tune GIAI PHONG dung luong cho EHR.')
R.to_csv(os.path.join(RES38B, 'main_table.csv'), index=False)
D.to_csv(os.path.join(RES38B, 'paired_diff.csv'), index=False)

## C8 · Tráo nhân quả — bắt buộc cho mọi arm có EHR

`06b` đo `corr(logit) = 0.999934` khi tráo EHR sang bệnh nhân khác; đó là bằng chứng mạnh nhất của
họ rằng token EHR **không được dùng**. mAP tăng mà `corr` vẫn ~1.0 thì mức tăng **không đến từ EHR**.

In [ ]:
sk = f's{SEEDS[0]}'
ck = torch.load(os.path.join(CKPT, f'{sk}.pt'), map_location=dev, weights_only=False)
for k in BACKBONES: MODS[k].load_state_dict(ck['mods'][k]); ETKS[k].load_state_dict(ck['etks'][k])
rs = np.random.default_rng(0); perm = rs.permutation(len(I_VA))
P0 = predict(MODS, ETKS, I_VA); P1 = predict(MODS, ETKS, I_VA, perm=I_VA[perm])
lg = lambda p: np.log(p.clip(1e-6, 1-1e-6) / (1 - p.clip(1e-6, 1-1e-6)))
print(f'=== TRAO NHAN QUA ({sk}) — EHR sang benh nhan khac, NHAN giu nguyen ===')
print('  Doc HAI cot cung luc. `corr` mot minh QUA CHAT: mot arm co corr 0.99999 van co the mat')
print('  0.006 mAP khi trao — logit gan nhu khong doi nhung THU TU thi doi, ma AP la chi so XEP HANG.')
CONST = [t for t, c in ARMS.items() if c.get('const')]
for k in BACKBONES:
    print(f'\n--- {k} ---')
    for t in ARMS:
        if not ARMS[t]['ehr']: continue
        r = np.corrcoef(lg(P0[(k,t)]).ravel(), lg(P1[(k,t)]).ravel())[0, 1]
        d = float(np.nanmean(ap_cols(YM[VA], P1[(k,t)])) - np.nanmean(ap_cols(YM[VA], P0[(k,t)])))
        v = ('EHR co tac dong that' if d <= -0.002 else
             'EHR KHONG duoc dung'  if r > 0.9999 and d > -0.0005 else 'vung xam')
        print(f'  {t:<18} corr(logit) {r:.6f} | mAP khi trao {d:+.4f}   {v}')
        # doi chung `const` KHONG nhan EHR => trao phai KHONG doi gi. Lech = ro ri.
        if t in CONST:
            assert r > 1 - 1e-9 and abs(d) < 1e-9, \
                f'{k}/{t} la doi chung HANG SO nhung trao EHR van doi ket qua (corr {r:.9f}, d {d:+.6f}) -> ro ri'
print(f'\n  moc 06b: corr = 0.999934 (H6)')
print(f'  cong: doi chung hang so {CONST} bat bien duoi phep trao — dung nhu thiet ke')

## C9 · Ghi `REPORT_STAGE4_NB38B.md`

In [ ]:
L = []; w = L.append
w('# REPORT · `nb38b` — arm EHR trên backbone đã fine-tune'); w('')
w(f'`{RUN_TAG}` · backbone `ft` = `38a/{BB_RUN}` · `asl40` · `E={EPOCHS}` · **epoch cuối** · '
  f'BATCH {BATCH}×{ACCUM} · seeds {list(SEEDS)} · val {VA.sum():,} ảnh.'); w('')
w('| backbone | arm | mAP ảnh | biên độ | tail | `arm − mldec` | biên độ | cùng dấu |')
w('|---|---|---|---|---|---|---|---|')
for k in BACKBONES:
    for t in ARMS:
        _r = R[(R.bb == k) & (R.arm == t)]
        if not len(_r): continue
        _d = D[(D.bb == k) & (D.arm == t)]
        w(f'| `{k}` | `{t}` | {_r.mAP_anh.mean():.4f} | {_r.mAP_anh.max()-_r.mAP_anh.min():.4f} | '
          f'{_r["tail"].mean():.4f} | ' +
          (f'{_d.d_anh.iloc[0]:+.4f} | {_d.bien_do.iloc[0]:.4f} | {"co" if _d.cung_dau.iloc[0] else "KHONG"} |'
           if len(_d) else '— | — | — |'))
w('')
w('**Chỉ `arm − mldec` được đọc** — các arm dùng chung một forward nên hiệu của chúng là **ghép cặp**, '
  'phương sai nhỏ hơn hẳn con số tuyệt đối. Một hiệu chỉ đáng tin khi **cùng dấu qua mọi seed** VÀ '
  '`|d| > biên độ`.'); w('')
w('## Những gì kết quả này KHÔNG loại trừ'); w('')
w(f'- **Backbone.** Chỉ `38a/{BB_RUN}` (`E=3`, `LR 1e-3`). `E` của nó được chọn bằng đường cong '
  '**val** của `a1`; `a4` sau đó cho thấy lát chọn và val đồng ý về epoch đỉnh, nhưng món nợ vẫn ghi.')
w('- **Biểu diễn EHR.** One-hot top-K vi khuẩn là biểu diễn sai cho biến phân loại đuôi dài. '
  '`EmbeddingBag` (mỗi vi khuẩn một vector, cộng các vector có mặt) là đúng và cohort cũ đã dùng — '
  'nhưng nó nằm ở `29b`, không ở đây. Đây là **thay đổi duy nhất còn lại có khả năng đổi bậc độ lớn**.')
w('- **Fine-tune có EHR trong vòng lặp** (`39`, B2d) — ở đây backbone đông cứng nên EHR không thể '
  'định hình biểu diễn, chỉ có thể điều biến đầu ra.')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo định '
  'nghĩa. Không con số nào ở đây nói gì về tiên lượng, tử vong, chuyển ICU, tái nhập viện.')
TXT = '\n'.join(L)
open(os.path.join(RES38B, 'REPORT_STAGE4_NB38B.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)